# 10 - Stdlib, Regex, Typing, Concurrency, Testing & Debugging

The last notebook, and the hardest. Everything here is about the code *around* your code: the standard library you should not be rewriting, regular expressions, type hints, the three flavours of concurrency, and the two skills that decide how fast you fix things — writing tests and reading tracebacks.

`collections`, `itertools` and `functools` are the three modules that most reliably delete lines from a codebase. Regex is where an afternoon disappears if you have never met greedy quantifiers or the difference between `match` and `search`. The concurrency questions are deliberately about *results*, not speed — the GIL decides whether threads help you at all, and knowing which side of that line your workload sits on is worth more than any API detail.

Q-400 asks you to rebuild `functools.lru_cache`. It is the last question of the course on purpose: closures, decorators, `OrderedDict`, and an eviction policy, in about twenty lines. If you can write it, you can read the standard library.

---

**How to use this notebook**

1. Read the question. Write your answer in the code cell beneath it.
2. Run the cell. `check(...)` tells you PASS or FAIL, and shows the failing case.
3. Stuck? Open **Hint** first. Open **Solution** only after a genuine attempt.
4. Call `progress()` any time to see how far you have got.

**Difficulty labels** — questions run easiest to hardest down the notebook.

| | |
|---|---|
| L1 Extremely Easy | one concept, one line |
| L2 Easy | one concept applied, a small edge case |
| L3 Intermediate | 2-3 concepts combined, or a non-obvious edge case |
| L4 Somewhat Difficult | a design choice, correctness needs care |
| L5 Hard | multi-concept, subtle failure modes |


In [ ]:
# Run this cell first.
# Finds the repo root by walking up from wherever Jupyter was started, so this
# works whether you launched from the project root, from notebooks/, or
# installed the package with `pip install -e .`
import pathlib
import sys

if "sprintcheck" not in sys.modules:
    here = pathlib.Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "sprintcheck" / "__init__.py").exists():
            sys.path.insert(0, str(folder))
            break

try:
    from sprintcheck import check, progress, reset
except ModuleNotFoundError:
    raise SystemExit(
        "Could not find the sprintcheck package.\n"
        "Open this notebook from inside the cloned repository, or run\n"
        "    pip install -e .\n"
        "from the project root and restart the kernel."
    )

print("Ready. Solve a question, then run check('Q-001', your_answer).")
print("Call progress() any time to see how far you have got.")


---

## L1 - Extremely Easy

### Q-359 - L1 Extremely Easy - Modules

Import the `math` module and write `circle_area(r)` that returns the area of a circle of radius `r`, rounded to 2 decimal places.

`circle_area(1)` returns `3.14`. Use `math.pi` — do not type the digits yourself.

<details><summary>Hint</summary>

`import math` makes the whole module available under one name; you reach inside it with a dot. The constant you want is spelled in lowercase.

</details>

<details><summary>Solution</summary>

```python
import math


def circle_area(r):
    return round(math.pi * r ** 2, 2)
```

**Why:** `import math` binds one name and keeps every constant behind it, which is why `math.pi` reads unambiguously while `from math import *` would let a later assignment to `pi` silently shadow it. Modules are imported once per process and cached in `sys.modules`, so a second `import math` anywhere in your program costs nothing.

</details>

In [ ]:
import math


def circle_area(r):
    ...

check('Q-359', circle_area)

### Q-360 - L1 Extremely Easy - Collections

Write `tally(words)` that counts how often each word appears and returns the result as a **plain `dict`**.

Use `collections.Counter` to do the counting, then convert. The checker is strict on type: a `Counter` will not pass where a `dict` is expected, even though the two compare equal.

<details><summary>Hint</summary>

`Counter(words)` does the whole count in one call. The only remaining job is changing the wrapper — the builtin that builds a dict from any mapping is the one you want.

</details>

<details><summary>Solution</summary>

```python
from collections import Counter


def tally(words):
    return dict(Counter(words))
```

**Why:** `Counter` is a `dict` subclass, so `==` says yes but `type(x) is dict` says no — and code that later does `json.dumps` or an `isinstance` check can care about the difference. Converting at the boundary of your function is the habit: use the rich type internally, hand back the plain one.

</details>

In [ ]:
from collections import Counter


def tally(words):
    ...

check('Q-360', tally)

---

## L2 - Easy

### Q-361 - L2 Easy - Testing

Write `assert_close(a, b, tol)` — the kind of helper a test suite is built from.

It returns `None` when `abs(a - b) <= tol`, and raises `AssertionError` otherwise. Use the `assert` statement, and give it a message that names both values.

<details><summary>Hint</summary>

`assert <condition>, <message>` raises `AssertionError` with that message when the condition is falsy, and does nothing at all when it is truthy. A function that falls off the end returns `None` by itself.

</details>

<details><summary>Solution</summary>

```python
def assert_close(a, b, tol):
    assert abs(a - b) <= tol, f"{a} and {b} differ by more than {tol}"
```

**Why:** pytest rewrites plain `assert` statements so the failure message shows the actual operand values — which is why you write `assert x == y` and not `self.assertEqual`. The message you attach is for the case the values alone cannot explain; leaving it off is fine, leaving the comparison vague is not. Remember `python -O` strips every `assert`, so never put real logic in one.

</details>

In [ ]:
def assert_close(a, b, tol):
    ...

check('Q-361', assert_close)

### Q-362 - L2 Easy - Regex

Write `find_numbers(text)` that returns a **list of strings**: every run of consecutive digits in `text`, in order.

`find_numbers("a1b22")` returns `['1', '22']`. Return the matched text, not `int`s. Use `re.findall`.

<details><summary>Hint</summary>

`\d` matches one digit; a quantifier turns it into a run. Write the pattern as a raw string so the backslash reaches the regex engine intact.

</details>

<details><summary>Solution</summary>

```python
import re


def find_numbers(text):
    return re.findall(r"\d+", text)
```

**Why:** `re.findall` returns *all* non-overlapping matches as a list and never returns `None`, which makes it the easiest entry point into `re`. Always write patterns as raw strings: `"\d"` happens to survive today but Python now warns about unknown escapes, and `"\b"` really is a backspace character rather than a word boundary.

</details>

In [ ]:
import re


def find_numbers(text):
    ...

check('Q-362', find_numbers)

### Q-363 - L2 Easy - JSON

Write `to_json(obj)` that serialises `obj` to a JSON **string** with the keys sorted alphabetically.

`to_json({"b": 1, "a": 2})` returns `'{"a": 2, "b": 1}'` — note the default spacing, which you should not change.

<details><summary>Hint</summary>

One function in `json` turns an object into a string; its sibling reads a string back. The sorting is a keyword argument, not something you do yourself beforehand.

</details>

<details><summary>Solution</summary>

```python
import json


def to_json(obj):
    return json.dumps(obj, sort_keys=True)
```

**Why:** `dumps` makes a string and `dump` writes to a file — the trailing `s` is the only difference, and mixing them up is the single most common `json` mistake. `sort_keys=True` is what makes serialised output stable enough to diff or hash; without it you are at the mercy of insertion order.

</details>

In [ ]:
import json


def to_json(obj):
    ...

check('Q-363', to_json)

### Q-364 - L2 Easy - Datetime

Write `days_between(start, end)` where both arguments are ISO date strings like `"2024-03-01"`. Return the number of days from `start` to `end` as an **`int`**.

The result is negative when `end` comes first. Do not call `today()` or `now()` — everything you need is in the arguments.

<details><summary>Hint</summary>

`datetime.date` has a classmethod that parses exactly this format. Subtracting two dates gives you a `timedelta`, which carries the number you want in one of its attributes.

</details>

<details><summary>Solution</summary>

```python
from datetime import date


def days_between(start, end):
    return (date.fromisoformat(end) - date.fromisoformat(start)).days
```

**Why:** `date.fromisoformat` handles leap years for you — 2024-01-01 to 2024-03-01 is 60 days, not 59 — which is the whole reason never to do date arithmetic by multiplying out days per month. `timedelta.days` is a whole number of days and can be negative; `.seconds` is the *remainder* within a day, not a total, and reading it as one is a classic bug.

</details>

In [ ]:
from datetime import date


def days_between(start, end):
    ...

check('Q-364', days_between)

### Q-365 - L2 Easy - Random

Write `roll(n)` that returns a list of `n` dice rolls — integers from 1 to 6 inclusive.

**Seed the generator with `random.seed(0)` as the first thing the function does**, so the same call always gives the same list. Use `random.randint`.

<details><summary>Hint</summary>

`random.randint(a, b)` includes both ends, unlike almost everything else in Python. Reseed on every call, not once at import, or the second call will continue the stream instead of restarting it.

</details>

<details><summary>Solution</summary>

```python
import random


def roll(n):
    random.seed(0)
    return [random.randint(1, 6) for _ in range(n)]
```

**Why:** `random` is a deterministic pseudo-random generator: the seed fixes the entire stream, which is what makes a randomised test reproducible instead of a lottery. `randint(1, 6)` is inclusive on both ends while `randrange(1, 6)` stops at 5 — and for anything security-related you want `secrets`, never `random`.

</details>

In [ ]:
import random


def roll(n):
    ...

check('Q-365', roll)

### Q-366 - L2 Easy - Type Hints

Write `mean` with **full type annotations**:

```python
def mean(values: list[float]) -> float:
```

Return the arithmetic mean as a `float`, and `0.0` for an empty list. Use the builtin generic `list[float]` — not `typing.List`, which has been deprecated since 3.9.

<details><summary>Hint</summary>

The annotation goes after a colon on the parameter and after an arrow before the body's colon. Guard the empty case first, or the division will raise.

</details>

<details><summary>Solution</summary>

```python
def mean(values: list[float]) -> float:
    if not values:
        return 0.0
    return sum(values) / len(values)
```

**Why:** Annotations are stored on `mean.__annotations__` and are read by mypy, pyright and your editor — the interpreter itself never checks them (Q-377 makes that vivid). Since 3.9 the builtin containers are subscriptable directly, so `list[float]` needs no import at all.

</details>

In [ ]:
def mean(values: list[float]) -> float:
    ...

check('Q-366', mean)

---

## L3 - Intermediate

### Q-367 - L3 Intermediate - Collections

Write `group_by_initial(words)` that groups words by their lowercased first letter.

Return a **plain `dict`** mapping that letter to the list of words that start with it, each list in the order the words arrived. Build it with `collections.defaultdict(list)` and convert at the end.

`group_by_initial(["Ant", "ant"])` returns `{'a': ['Ant', 'ant']}`.

<details><summary>Hint</summary>

`defaultdict(list)` calls `list()` for you the first time a key is touched, so you can append immediately with no `if key not in d` dance. Remember the return type is asked to be plain.

</details>

<details><summary>Solution</summary>

```python
from collections import defaultdict


def group_by_initial(words):
    groups = defaultdict(list)
    for word in words:
        groups[word[0].lower()].append(word)
    return dict(groups)
```

**Why:** `defaultdict` removes the setdefault/`in` boilerplate that group-by code is otherwise made of, and because dicts preserve insertion order the groups come out in first-seen order for free. The catch is that *any* lookup creates the key — `groups["z"]` in a debug print silently adds an empty list — so convert to a plain `dict` before you hand the result out.

</details>

In [ ]:
from collections import defaultdict


def group_by_initial(words):
    ...

check('Q-367', group_by_initial)

### Q-368 - L3 Intermediate - Regex

Without running it, predict the output:

```python
import re
print(re.match(r"\d+", "abc123"))
print(re.search(r"\d+", "abc123").group())
print(bool(re.match(r"\d+", "123abc")))
```

Set `answer` to the three printed lines.

<details><summary>Hint</summary>

One of these two functions is allowed to start looking anywhere; the other is not. Neither of them requires the pattern to consume the *whole* string.

</details>

<details><summary>Solution</summary>

```python
answer = "None\n123\nTrue"
```

**Why:** `re.match` is anchored at position 0 and returns `None` the moment the pattern does not fit there, while `re.search` scans forward — the mismatch that produces the classic `AttributeError: 'NoneType' object has no attribute 'group'`. Neither anchors the *end*: `re.match(r"\d+", "123abc")` succeeds on the prefix, so use `re.fullmatch` when you mean 'the entire string'.

</details>

In [ ]:
answer = ...

check('Q-368', answer)

### Q-369 - L3 Intermediate - Regex

Write `parse_date(text)` that finds an ISO date anywhere in `text` — four digits, a dash, two digits, a dash, two digits — using **named groups** called `year`, `month` and `day`.

Return a **plain `dict`** of the three captured **strings** (keep the leading zeros), or `None` when there is no such date.

<details><summary>Hint</summary>

A named group is written `(?P<name>...)`. A match object can hand you every named group at once as a dict — look for the method whose name says exactly that. And you must search, not match.

</details>

<details><summary>Solution</summary>

```python
import re

DATE_RE = re.compile(r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})")


def parse_date(text):
    found = DATE_RE.search(text)
    if found is None:
        return None
    return found.groupdict()
```

**Why:** Named groups turn a match into self-documenting code: `m["month"]` survives someone inserting a group in front of it, `m.group(2)` does not. `groupdict()` hands back a real `dict` of strings — the regex engine never converts types, so `"03"` stays a string until you call `int` on it.

</details>

In [ ]:
import re


def parse_date(text):
    ...

check('Q-369', parse_date)

### Q-370 - L3 Intermediate - Itertools

Write `pairs(items)` that returns a **list of 2-tuples**: every unordered pair of distinct positions, in the order `itertools.combinations` produces them.

`pairs([1, 2, 3])` returns `[(1, 2), (1, 3), (2, 3)]`. A list with fewer than two items gives `[]`.

<details><summary>Hint</summary>

One `itertools` function produces exactly this and takes the group size as its second argument. It returns an iterator, so the result needs wrapping before you hand it back.

</details>

<details><summary>Solution</summary>

```python
from itertools import combinations


def pairs(items):
    return list(combinations(items, 2))
```

**Why:** `combinations` treats positions, not values, as distinct and emits them in input order — so it never produces both `(a, b)` and `(b, a)`, which is what separates it from `permutations`. Like every `itertools` function it returns a lazy iterator that is consumed once, so `list()` is not decoration.

</details>

In [ ]:
from itertools import combinations


def pairs(items):
    ...

check('Q-370', pairs)

### Q-371 - L3 Intermediate - Functools

Write `product(numbers)` that multiplies every number together and returns the result, using `functools.reduce`.

The product of an empty list is `1`. Do not write an explicit loop.

<details><summary>Hint</summary>

`reduce(func, iterable, initial)` folds left, two items at a time. The third argument is not optional here — think about what `reduce` does when handed an empty sequence and nothing to start from.

</details>

<details><summary>Solution</summary>

```python
from functools import reduce


def product(numbers):
    return reduce(lambda a, b: a * b, numbers, 1)
```

**Why:** Without the `1`, `reduce` raises `TypeError` on an empty iterable — passing an explicit identity is what makes a fold total rather than a special case. `reduce` moved out of builtins in Python 3 because an explicit loop is usually clearer; reach for it when the operation genuinely is a fold, and prefer `math.prod` for this particular one.

</details>

In [ ]:
from functools import reduce


def product(numbers):
    ...

check('Q-371', product)

### Q-372 - L3 Intermediate - Pathlib

Write `describe_path(text)` that takes a POSIX path as a string and returns the 4-tuple `(name, stem, suffix, parent)` — all four as **strings**.

`describe_path("/var/log/app.log")` returns `('app.log', 'app', '.log', '/var/log')`.

Use `pathlib.PurePosixPath`, which does pure string surgery and never touches the disk. Note `parent` must be a `str`, not a path object.

<details><summary>Hint</summary>

Three of the four are attributes with exactly those names. The fourth is also an attribute, but it hands you a path object — convert it.

</details>

<details><summary>Solution</summary>

```python
from pathlib import PurePosixPath


def describe_path(text):
    path = PurePosixPath(text)
    return (path.name, path.stem, path.suffix, str(path.parent))
```

**Why:** `pathlib` replaces a pile of `os.path` calls with attributes that read like the thing you want, and the `Pure*` variants are safe in tests because they never stat anything. Watch the double extension: the suffix of `notes.tar.gz` is `.gz` and the stem is `notes.tar` — `suffixes` (plural) is what gives you both.

</details>

In [ ]:
from pathlib import PurePosixPath


def describe_path(text):
    ...

check('Q-372', describe_path)

### Q-373 - L3 Intermediate - Collections

Write `last_n(items, n)` that returns a **list** of the last `n` items, in order — fewer than `n` if the input is shorter, and `[]` when `n` is `0`.

Feed the items one at a time into a `collections.deque(maxlen=n)` and convert it at the end. The checker is strict: a `deque` is not a `list`.

<details><summary>Hint</summary>

A deque with a `maxlen` drops from the opposite end automatically as you append, so no slicing or length check is needed. `extend` does the feeding in one call.

</details>

<details><summary>Solution</summary>

```python
from collections import deque


def last_n(items, n):
    window = deque(maxlen=n)
    window.extend(items)
    return list(window)
```

**Why:** A bounded `deque` is the standard ring buffer: appending past `maxlen` discards from the far end in O(1), which `items[-n:]` cannot do when the items arrive from a stream you never want to hold in full. Beware `n == 0`, which makes a deque that discards everything — and beware `items[-0:]`, which returns the *whole* list rather than nothing.

</details>

In [ ]:
from collections import deque


def last_n(items, n):
    ...

check('Q-373', last_n)

### Q-374 - L3 Intermediate - Concurrency

No code to run — decide `True` or `False` for each claim, one per line, in order:

1. Splitting a pure-Python CPU-bound loop across 4 `threading.Thread`s makes it roughly 4x faster.
2. Splitting 4 blocking network downloads across 4 `threading.Thread`s makes them finish sooner.
3. Splitting the same pure-Python CPU-bound loop across 4 processes with `multiprocessing` makes it faster.

Set `answer` to the three words, e.g. `answer = "True\nTrue\nTrue"`.

<details><summary>Hint</summary>

One lock in the interpreter decides all three. Ask what a thread is holding while it executes bytecode, and whether it is still holding it while it waits on a socket.

</details>

<details><summary>Solution</summary>

```python
answer = "False\nTrue\nTrue"
```

**Why:** CPython's Global Interpreter Lock lets exactly one thread execute bytecode at a time, so threads cannot add CPU throughput — but a thread *releases* the GIL while it blocks on I/O, which is why waiting in parallel works beautifully. Processes each get their own interpreter and their own GIL, at the cost of pickling everything that crosses between them. Hence the rule: threads (or `asyncio`) for I/O-bound work, processes for CPU-bound work.

</details>

In [ ]:
answer = ...

check('Q-374', answer)

### Q-375 - L3 Intermediate - Testing

`@pytest.mark.parametrize("a,b,expected", CASES)` needs `CASES` to be a list of tuples, one tuple per run, with the arguments in the same order as the names in the string.

Write `build_cases(func, arg_pairs)`: given a two-argument function and a list of `(a, b)` tuples, return the **list of 3-tuples** `(a, b, func(a, b))` — a parametrize table with the expected values filled in by calling the function.

<details><summary>Hint</summary>

One comprehension: unpack each pair, and put the call in the third slot. The shape of the result is what parametrize consumes, so keep it tuples inside a list.

</details>

<details><summary>Solution</summary>

```python
def build_cases(func, arg_pairs):
    return [(a, b, func(a, b)) for a, b in arg_pairs]
```

**Why:** `parametrize` turns one test body into N independent tests, so a failure names the exact row instead of stopping at the first bad input — that is why it beats a `for` loop inside a single test. Generating the expected column by calling the function under test is a *characterisation* test: it pins current behaviour before a refactor, but it can never tell you that behaviour was right.

</details>

In [ ]:
def build_cases(func, arg_pairs):
    ...

check('Q-375', build_cases)

### Q-376 - L3 Intermediate - Debugging

Here is a traceback exactly as Python printed it:

```
Traceback (most recent call last):
  File "app.py", line 9, in <module>
    main()
  File "app.py", line 6, in main
    return totals["gamma"]
           ~~~~~~^^^^^^^^^
KeyError: 'gamma'
```

Set `answer` to two lines: the name of the function where the error actually happened, then the `app.py` line number of the statement that raised. For example `answer = "main\n9"`.

<details><summary>Hint</summary>

The header says *most recent call last*, so read the frames from the bottom up. The caret line underneath points at the sub-expression that blew up within its frame.

</details>

<details><summary>Solution</summary>

```python
answer = "main\n6"
```

**Why:** A traceback is a call stack printed oldest-first, so the frame nearest the exception message — the bottom one — is where execution actually stopped; everything above it is just how you got there. Since 3.11 the `~~~^^^` markers narrow it further to the failing sub-expression, which is what tells you it was the subscript and not the `return`.

</details>

In [ ]:
answer = ...

check('Q-376', answer)

### Q-377 - L3 Intermediate - Type Hints

Without running it, predict the output:

```python
def double(n: int) -> int:
    return n * 2

print(double("ab"))
print(double.__annotations__["n"] is int)
```

Set `answer` to the two printed lines.

<details><summary>Hint</summary>

Ask what the interpreter *does* with an annotation when the function is called. Then ask what `*` means between a string and an integer.

</details>

<details><summary>Solution</summary>

```python
answer = "abab\nTrue"
```

**Why:** Annotations are metadata, not a contract: nothing checks them at call time, so `double("ab")` happily returns `'abab'` because `str * int` is valid Python. They are stored on `__annotations__` for mypy, pyright and your editor to read — if you want enforcement at runtime you have to opt in, with `pydantic`, `beartype`, or a check you write yourself.

</details>

In [ ]:
answer = ...

check('Q-377', answer)

---

## L4 - Somewhat Difficult

### Q-378 - L4 Somewhat Difficult - Regex

Without running it, predict the output:

```python
import re
text = "<a><b>"
print(re.findall(r"<.*>", text))
print(re.findall(r"<.*?>", text))
```

Set `answer` to the two printed lines, written exactly as Python prints a list of strings.

<details><summary>Hint</summary>

`*` takes as much as it can and only gives characters back when the rest of the pattern fails. The `?` after it reverses that appetite.

</details>

<details><summary>Solution</summary>

```python
answer = "['<a><b>']\n['<a>', '<b>']"
```

**Why:** `.*` is greedy: it swallows to the end of the string and then backtracks to the *last* `>`, producing one match that spans both tags. `.*?` is lazy and stops at the first `>`, giving the two matches you meant. The sharper fix for markup-like input is a negated class, `<[^>]*>`, which cannot overshoot at all and so needs no backtracking.

</details>

In [ ]:
answer = ...

check('Q-378', answer)

### Q-379 - L4 Somewhat Difficult - Regex

Write `mask_digits(text)` that replaces every run of digits with the same number of `#` characters, leaving everything else alone.

`mask_digits("id 4821 x7")` returns `'id #### x#'`.

Use `re.sub` with a **function** as the replacement, not a fixed string — the length of each replacement depends on the match.

<details><summary>Hint</summary>

`re.sub(pattern, repl, text)` accepts a callable for `repl`; it is handed the match object and must return the replacement string. The match knows its own text.

</details>

<details><summary>Solution</summary>

```python
import re


def mask_digits(text):
    return re.sub(r"\d+", lambda m: "#" * len(m.group()), text)
```

**Why:** A callable `repl` is the escape hatch for any substitution that has to compute something, and it sidesteps escaping entirely — the string it returns is used literally, so a `\1` or `\g<0>` in it means nothing. With a *string* `repl` the opposite is true: backslashes are interpreted as group references, which is how a stray `\1` in replacement text turns into an `error: invalid group reference`.

</details>

In [ ]:
import re


def mask_digits(text):
    ...

check('Q-379', mask_digits)

### Q-380 - L4 Somewhat Difficult - Regex

Write `settings_from(text)` that pulls `key=value` assignments out of a string and returns them as a **list of `(key, value)` tuples of strings**, in order.

`settings_from("a=1;b=2")` returns `[('a', '1'), ('b', '2')]`. Keys and values are runs of word characters; anything else is a separator and is ignored. Use `re.findall` with two capturing groups.

<details><summary>Hint</summary>

What `findall` gives back depends on how many groups your pattern has: none, one, or more than one all behave differently. Two groups is exactly the case that yields tuples.

</details>

<details><summary>Solution</summary>

```python
import re


def settings_from(text):
    return re.findall(r"(\w+)=(\w+)", text)
```

**Why:** `findall` has three modes and this is the third: with no groups it returns whole matches, with one group it returns that group's text, and with two or more it returns a tuple per match. Adding a group to an existing pattern therefore changes the *shape* of your results — which is what non-capturing `(?:...)` exists to prevent.

</details>

In [ ]:
import re


def settings_from(text):
    ...

check('Q-380', settings_from)

### Q-381 - L4 Somewhat Difficult - Concurrency

Write `lengths(texts)` that returns a **list of `int`s** — the length of each string, in the same order as the input.

Do the work through a `concurrent.futures.ThreadPoolExecutor` with `max_workers=4`, using its `.map()`. Use the executor as a context manager so it shuts down cleanly, and convert the result to a list.

<details><summary>Hint</summary>

`pool.map(func, iterable)` mirrors the builtin `map` but runs the calls in worker threads. It returns a lazy iterator, and leaving the `with` block waits for every task to finish.

</details>

<details><summary>Solution</summary>

```python
from concurrent.futures import ThreadPoolExecutor


def lengths(texts):
    with ThreadPoolExecutor(max_workers=4) as pool:
        return list(pool.map(len, texts))
```

**Why:** `Executor.map` keeps results in **input order** no matter which worker finishes first, which is what makes it a drop-in for `map` — use `as_completed` when you want them in completion order instead. It also re-raises a worker's exception at the point you consume that item, so a failure inside a thread is never silently lost. Exiting the `with` block calls `shutdown(wait=True)`.

</details>

In [ ]:
from concurrent.futures import ThreadPoolExecutor


def lengths(texts):
    ...

check('Q-381', lengths)

### Q-382 - L4 Somewhat Difficult - Collections

Write `centroid(pairs)` where `pairs` is a list of `(x, y)` tuples.

Define `Point = namedtuple("Point", "x y")`, wrap each pair in one, and compute the average position using **attribute access** (`p.x`, `p.y`) rather than indexing.

Return a **plain `dict`** `{'x': ..., 'y': ...}` with both values as `float`s, or `None` for an empty list.

<details><summary>Hint</summary>

`Point(*pair)` unpacks a 2-tuple straight into the two fields. Dividing by the count with `/` already gives you floats, so no cast is needed.

</details>

<details><summary>Solution</summary>

```python
from collections import namedtuple

Point = namedtuple("Point", "x y")


def centroid(pairs):
    points = [Point(*pair) for pair in pairs]
    if not points:
        return None
    n = len(points)
    return {
        "x": sum(p.x for p in points) / n,
        "y": sum(p.y for p in points) / n,
    }
```

**Why:** A `namedtuple` *is* a tuple — same memory, same unpacking, still hashable and immutable — but `p.x` says what `p[0]` only implies, which is the whole value. Because it is a tuple it also compares equal to a plain one, so it slots into existing code with no changes; when you want defaults, mutability or methods, graduate to a `dataclass`.

</details>

In [ ]:
from collections import namedtuple

Point = namedtuple("Point", "x y")


def centroid(pairs):
    ...

check('Q-382', centroid)

### Q-383 - L4 Somewhat Difficult - Collections

Without running it, predict the output:

```python
from collections import Counter
a = Counter(a=3, b=1)
b = Counter(a=3, b=2)
print(a - b)
print(a + b)
print((a - b)["a"])
```

Set `answer` to the three printed lines, each written exactly as Python would show it.

<details><summary>Hint</summary>

`Counter` arithmetic is modelled on *multisets*, where a count below one means the element simply is not there. The third line asks a different question from the first two — what does a `Counter` do for a missing key?

</details>

<details><summary>Solution</summary>

```python
answer = "Counter()\nCounter({'a': 6, 'b': 3})\n0"
```

**Why:** `+` and `-` between Counters **discard every count that is not positive**, so `a - b` drops `a` (3-3 = 0) and `b` (1-2 = -1) and leaves an empty Counter — the classic surprise when you use subtraction to compute a diff and the zeros vanish. Use `a.subtract(b)`, which mutates in place and keeps zero and negative counts, when you need them. Line three is the other half: indexing a missing key returns `0` rather than raising, and unlike `defaultdict` it does not insert the key.

</details>

In [ ]:
answer = ...

check('Q-383', answer)

### Q-384 - L4 Somewhat Difficult - Itertools

Write `group_lengths(words)` that groups words by their length using `itertools.groupby`.

Return a **plain `dict`** mapping each length (an `int`) to the list of words of that length, with the words in their original relative order.

`group_lengths(["bb", "a", "ccc", "dd"])` returns `{1: ['a'], 2: ['bb', 'dd'], 3: ['ccc']}`. There is one preparation step `groupby` requires that the prompt is deliberately not telling you.

<details><summary>Hint</summary>

`groupby` only ever compares each item with the one immediately before it. That means it can only find a group if equal keys are already adjacent — so something has to happen to the input first, with the same key function.

</details>

<details><summary>Solution</summary>

```python
from itertools import groupby


def group_lengths(words):
    ordered = sorted(words, key=len)
    return {k: list(g) for k, g in groupby(ordered, key=len)}
```

**Why:** `itertools.groupby` breaks a stream into *runs* of consecutive equal keys — it is not SQL's GROUP BY — so unsorted input silently produces several groups with the same key and the last one wins in a dict. Sorting with the identical key function is the fix, and because `sorted` is stable the words keep their original relative order. The second trap: each group is a lazy iterator tied to the shared source, so it is empty as soon as you advance to the next group — `list(g)` has to happen now, not later.

</details>

In [ ]:
from itertools import groupby


def group_lengths(words):
    ...

check('Q-384', group_lengths)

### Q-385 - L4 Somewhat Difficult - Functools

Write `fib(n)` returning the nth Fibonacci number, with `fib(0) == 0` and `fib(1) == 1`.

It must be **recursive** — `fib` has to call itself — and it must be decorated with `@functools.lru_cache(maxsize=None)`.

One of the test cases is `fib(35)`. Without the cache that is about 30 million calls and takes seconds; with it, 36.

<details><summary>Hint</summary>

Write the naive two-call recursion first, then put one decorator line above the `def`. The decorator needs the arguments to be hashable, which plain `int`s are.

</details>

<details><summary>Solution</summary>

```python
from functools import lru_cache


@lru_cache(maxsize=None)
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)
```

**Why:** `lru_cache` keys on the *arguments*, so the exponential recursion collapses to linear the moment the decorator is in place — the recursive calls go through the wrapper, which is what makes memoising a recursive function work at all. Two real costs to know: arguments must be hashable (no lists), and the cache holds a strong reference to every argument — so `@lru_cache` on a *method* keeps `self` alive forever and quietly leaks every instance. Cache the free function, or use `functools.cached_property` on the instance.

</details>

In [ ]:
from functools import lru_cache


@lru_cache(maxsize=None)
def fib(n):
    ...

check('Q-385', fib)

### Q-386 - L4 Somewhat Difficult - Functools

Without running it, predict the output:

```python
from functools import partial

def add_item(item, bucket=[]):
    bucket.append(item)
    return bucket

push = partial(add_item)
print(push("a"))
print(push("b"))
```

Set `answer` to the two printed lines.

<details><summary>Hint</summary>

Ask when the default `[]` is evaluated: once, or once per call? `partial` changes nothing about that — it only remembers arguments you *did* pass.

</details>

<details><summary>Solution</summary>

```python
answer = "['a']\n['a', 'b']"
```

**Why:** A default argument is evaluated **once**, when the `def` executes, and that single list is reused by every call that does not override it — so the bucket accumulates across calls forever. `partial` does not insulate you: it stores the arguments it was given and forwards the rest, leaving the function's own defaults exactly as they were. The fix is always the same: default to `None` and build the container inside the body.

</details>

In [ ]:
answer = ...

check('Q-386', answer)

### Q-387 - L4 Somewhat Difficult - File I/O

Write `wc(text)` — a `wc`-style counter that returns the tuple `(line_count, word_count)`.

The catch: it must read `text` through a **file-like object**, not by splitting the string directly. Wrap it in `io.StringIO` and open it with a `with` statement, then use `.readlines()`.

A line is a line as `readlines()` sees it; words are whitespace-separated. `wc("")` is `(0, 0)`.

<details><summary>Hint</summary>

`io.StringIO(text)` is an in-memory text file and supports the same context manager protocol as a real one. Counting words per line and summing is easier than one pass over the whole text.

</details>

<details><summary>Solution</summary>

```python
import io


def wc(text):
    with io.StringIO(text) as handle:
        lines = handle.readlines()
    return (len(lines), sum(len(line.split()) for line in lines))
```

**Why:** `with` guarantees the file is closed on every path out — exception included — which is the difference between a leaked handle and a program that runs for a month. Writing against `io.StringIO` instead of a real path is also the standard way to make I/O code testable: the function only needs *something file-shaped*, so tests need no `tmp_path`, no cleanup, and no disk. Note `.readlines()` keeps the `\n` on each line, which is why `.split()` and not `len(line)` is doing the word counting.

</details>

In [ ]:
import io


def wc(text):
    ...

check('Q-387', wc)

### Q-388 - L4 Somewhat Difficult - Datetime

Without running it, predict the output:

```python
from datetime import datetime, timezone

naive = datetime(2024, 1, 1, 12, 0)
aware = datetime(2024, 1, 1, 12, 0, tzinfo=timezone.utc)
print(naive == aware)
try:
    print(naive < aware)
except TypeError as exc:
    print(type(exc).__name__)
```

Set `answer` to the two printed lines.

<details><summary>Hint</summary>

One of these two datetimes knows what part of the world it is in; the other does not. Equality and ordering do not have to agree about what to do with that — one of them can fall back on 'not the same thing'.

</details>

<details><summary>Solution</summary>

```python
answer = "False\nTypeError"
```

**Why:** A naive datetime carries no `tzinfo`, so Python refuses to *order* it against an aware one — there is no defensible answer — and raises `TypeError: can't compare offset-naive and offset-aware datetimes`. Equality gets an exemption and simply returns `False` instead of raising, which is worse in practice: the bug sails through `==` and only detonates on the first `<`, `sorted()` or `min()`. Pick one convention per program; aware-UTC everywhere (`datetime.now(timezone.utc)`) is the one that scales.

</details>

In [ ]:
answer = ...

check('Q-388', answer)

### Q-389 - L4 Somewhat Difficult - Type Hints

Declare a `TypedDict` describing a user record:

```python
class User(TypedDict):
    name: str
    age: int
```

Then write `make_user(name, age)` that returns such a record. Raise `ValueError` when `age` is negative.

What comes back is an ordinary `dict` at runtime — the checker compares it as one.

<details><summary>Hint</summary>

A `TypedDict` is a type-checker construct, not a class you instantiate with attributes: build a normal dict literal with those keys. The validation is yours to write — the annotation will not do it.

</details>

<details><summary>Solution</summary>

```python
from typing import TypedDict


class User(TypedDict):
    name: str
    age: int


def make_user(name, age):
    if age < 0:
        raise ValueError("age must not be negative")
    user: User = {"name": name, "age": age}
    return user
```

**Why:** `TypedDict` gives a plain dict a *structural* type, so mypy flags a misspelled key or an `int` where a `str` belongs while the runtime object stays exactly a dict — no class, no overhead, and JSON-serialisable as-is. That is also its limit: nothing validates the actual data at runtime, which is why the `ValueError` has to be written by hand (and why `pydantic` exists for data arriving from outside your program).

</details>

In [ ]:
from typing import TypedDict


class User(TypedDict):
    name: str
    age: int


def make_user(name, age):
    ...

check('Q-389', make_user)

### Q-390 - L4 Somewhat Difficult - Type Hints

Write `apply_or` with this exact signature:

```python
def apply_or(
    func: Optional[Callable[[int], int]], value: int, default: int
) -> int:
```

Return `func(value)` when `func` is given, and `default` when it is `None`. Import `Callable` and `Optional` from `typing`.

Test for `None` with `is`, not with truthiness.

<details><summary>Hint</summary>

`Callable[[int], int]` means 'takes one int, returns an int' — the first bracket is the argument *list*. `Optional[X]` is shorthand for `X | None`, which is exactly the case your body has to branch on.

</details>

<details><summary>Solution</summary>

```python
from typing import Callable, Optional


def apply_or(
    func: Optional[Callable[[int], int]], value: int, default: int
) -> int:
    if func is None:
        return default
    return func(value)
```

**Why:** `Optional[X]` does not mean 'may be omitted' — it means the value may be `None`, and a type checker will demand you narrow it before calling. Narrowing with `if func is None` is what does that; `if not func:` would compile but is wrong in principle, since a callable object can define `__bool__`. Modern code writes the same type as `Callable[[int], int] | None`, no import needed.

</details>

In [ ]:
from typing import Callable, Optional


def apply_or(
    func: Optional[Callable[[int], int]], value: int, default: int
) -> int:
    ...

check('Q-390', apply_or)

### Q-391 - L4 Somewhat Difficult - Concurrency

Write `parallel_count(n_threads, per_thread)` that starts `n_threads` threads, has each one increment a single shared counter `per_thread` times, waits for all of them, and returns the final count as an `int`.

Guard the increment with a `threading.Lock` so the answer is always exactly `n_threads * per_thread`. Start every thread before joining any of them.

<details><summary>Hint</summary>

`with lock:` around the increment is the whole synchronisation. Two separate loops — one to `start()`, one to `join()` — is what makes them run concurrently instead of one after another.

</details>

<details><summary>Solution</summary>

```python
import threading


def parallel_count(n_threads, per_thread):
    total = 0
    lock = threading.Lock()

    def worker():
        nonlocal total
        for _ in range(per_thread):
            with lock:
                total += 1

    threads = [threading.Thread(target=worker) for _ in range(n_threads)]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    return total
```

**Why:** `total += 1` is three bytecodes — load, add, store — and a thread switch between the load and the store loses an update, so the GIL protects the interpreter's own state but not yours. The lock makes the read-modify-write atomic; `with lock` releases it even if the body raises, which a manual `acquire()`/`release()` pair does not. Starting in one loop and joining in a second matters: `start(); join()` in the same loop runs them strictly sequentially.

</details>

In [ ]:
import threading


def parallel_count(n_threads, per_thread):
    ...

check('Q-391', parallel_count)

### Q-392 - L4 Somewhat Difficult - Concurrency

Write `fetch_all(values)` — an ordinary **synchronous** function that returns a **list** with each value doubled, but does the work with `asyncio`.

Define an `async def` coroutine that doubles one value (with an `await asyncio.sleep(0)` in it to stand in for I/O), gather them all with `asyncio.gather`, and drive the whole thing from `fetch_all` with `asyncio.run(...)`.

`fetch_all` itself must not be `async` — the caller sees a plain function.

<details><summary>Hint</summary>

`asyncio.run(coro)` is the single bridge from sync code into the event loop: it creates a loop, runs the coroutine to completion and closes the loop. `gather` takes the coroutines as separate arguments, so unpack with `*`.

</details>

<details><summary>Solution</summary>

```python
import asyncio


async def _double(value):
    await asyncio.sleep(0)
    return value * 2


async def _main(values):
    return await asyncio.gather(*(_double(v) for v in values))


def fetch_all(values):
    return asyncio.run(_main(values))
```

**Why:** Calling an `async def` gives you a *coroutine object* and runs none of the body — the most common asyncio mistake is forgetting the `await` and wondering why nothing happened. `asyncio.gather` schedules them all concurrently and returns their results **in argument order**, not completion order. `asyncio.run` is the only entry point you should use from sync code, and it must not be called while a loop is already running (which is why it fails inside Jupyter, where a loop already exists).

</details>

In [ ]:
import asyncio


def fetch_all(values):
    ...

check('Q-392', fetch_all)

### Q-393 - L4 Somewhat Difficult - Testing

A pytest *fixture* is a factory: every test that asks for it gets a freshly built object, never the one the previous test scribbled on.

Write `run_with_fixture(factory, items)` that models this. For each item, call `factory()` to build a brand-new container, append the item to it, and collect that container. Return the **list of containers**.

`run_with_fixture(list, [1, 2])` returns `[[1], [2]]` — not `[[1, 2], [1, 2]]`. Call `factory()` inside the loop, not once before it.

<details><summary>Hint</summary>

The whole question is where the `factory()` call goes. Hoisting it above the loop is the bug you are demonstrating, not an optimisation.

</details>

<details><summary>Solution</summary>

```python
def run_with_fixture(factory, items):
    results = []
    for item in items:
        state = factory()
        state.append(item)
        results.append(state)
    return results
```

**Why:** This is why a fixture is a function and not a module-level object: state built once at import is shared by every test, so test 7 starts failing because test 3 mutated it, and the suite passes or fails depending on *order*. pytest's `scope=` argument is exactly the dial between the two — `function` scope (the default) rebuilds per test, `session` scope shares one and asks you to keep it read-only.

</details>

In [ ]:
def run_with_fixture(factory, items):
    ...

check('Q-393', run_with_fixture)

---

## L5 - Hard

### Q-394 - L5 Hard - Regex

Write `tokenize(source)` — a real tokenizer for a tiny expression language, built from **one compiled regex** with named alternatives.

Return a **list of `(kind, text)` tuples of strings**, where `kind` is one of:

- `NUMBER` — digits, optionally followed by `.` and more digits
- `NAME` — a letter or `_`, then letters/digits/underscores
- `OP` — one of `+ - * / =`

Whitespace is skipped and produces no token. Any other character raises `ValueError`.

`tokenize("x = 1")` returns `[('NAME', 'x'), ('OP', '='), ('NUMBER', '1')]`.

<details><summary>Hint</summary>

Join the alternatives with `|` into one pattern, each in its own named group, and walk it with `finditer`. A match object's `lastgroup` tells you which alternative won. Add a final catch-all group so an illegal character matches something you can reject, rather than being silently skipped.

</details>

<details><summary>Solution</summary>

```python
import re

TOKEN_RE = re.compile(
    r"(?P<NUMBER>\d+(?:\.\d+)?)"
    r"|(?P<NAME>[A-Za-z_]\w*)"
    r"|(?P<OP>[+\-*/=])"
    r"|(?P<SKIP>\s+)"
    r"|(?P<BAD>.)"
)


def tokenize(source):
    tokens = []
    for found in TOKEN_RE.finditer(source):
        kind = found.lastgroup
        if kind == "SKIP":
            continue
        if kind == "BAD":
            raise ValueError(f"unexpected character {found.group()!r}")
        tokens.append((kind, found.group()))
    return tokens
```

**Why:** This master-pattern approach is how `re` is used for real lexing — the standard library's own tokenizer example is built the same way — because one alternation with `lastgroup` beats trying each pattern in turn at every position. Two details carry the whole design. Alternation is **ordered and first-match-wins**, so `NUMBER` must precede `NAME` or `1x` tokenizes wrong, and `\d+(?:\.\d+)?` uses a non-capturing group so it does not disturb `lastgroup`. And the trailing `(?P<BAD>.)` is what converts silence into an error: `finditer` simply skips text nothing matches, so without a catch-all a junk character vanishes and you debug a mystery instead of reading an exception.

</details>

In [ ]:
import re


def tokenize(source):
    ...

check('Q-394', tokenize)

### Q-395 - L5 Hard - Itertools

Write `sliding_sums(values, size)` that returns a **list** of the sums of every contiguous window of length `size`.

`sliding_sums([1, 2, 3, 4], 2)` returns `[3, 5, 7]`. When the input is shorter than the window, return `[]`. A `size` of `0` or less raises `ValueError`.

The windows must be produced by a **generator** — define an inner `def` that `yield`s one sum at a time and materialise it with `list(...)` at the end. Slice each window with `itertools.islice`, not with `values[a:b]`.

<details><summary>Hint</summary>

Validate `size` before defining the generator, so the `ValueError` fires on the call and not on the first item pulled out. `range(len(values) - size + 1)` gives the valid start positions — and goes empty on its own when the input is too short.

</details>

<details><summary>Solution</summary>

```python
from itertools import islice


def sliding_sums(values, size):
    if size <= 0:
        raise ValueError("size must be positive")

    def windows():
        for start in range(len(values) - size + 1):
            yield sum(islice(values, start, start + size))

    return list(windows())
```

**Why:** Validating *before* the generator is defined is the real lesson: a `raise` inside a generator body does not happen when you call the function, it happens when something first iterates it — so a validating generator reports bad arguments at a stack frame far from the mistake. Splitting a plain wrapper around a generator core is the standard fix. `islice` also earns its place here: unlike a slice it works on any iterable and copies nothing, which is what lets the same shape run over a stream that never fits in memory.

</details>

In [ ]:
from itertools import islice


def sliding_sums(values, size):
    ...

check('Q-395', sliding_sums)

### Q-396 - L5 Hard - Type Hints

Write a **generic** container and drive it.

First `class Stack(Generic[T])` over `T = TypeVar("T")`, with `push(item: T) -> None`, `pop() -> T` (raising `IndexError` when empty) and `__len__`.

Then write `run_stack(ops)`: walk the list `ops`, and for each entry either pop (when the entry is the string `"pop"`) or push it. Return the tuple `(popped_values, remaining_length)` where `popped_values` is a **list** in the order they came off.

`run_stack([1, 2, "pop"])` returns `([2], 1)`. Popping an empty stack propagates the `IndexError`.

<details><summary>Hint</summary>

`Generic[T]` in the base list is what makes `Stack[int]` legal to write; the class body itself is ordinary Python. Annotate the internal list as `list[T]` so the element type flows through.

</details>

<details><summary>Solution</summary>

```python
from typing import Generic, TypeVar

T = TypeVar("T")


class Stack(Generic[T]):
    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)

    def pop(self) -> T:
        if not self._items:
            raise IndexError("pop from empty stack")
        return self._items.pop()

    def __len__(self) -> int:
        return len(self._items)


def run_stack(ops):
    stack: Stack[int] = Stack()
    popped = []
    for op in ops:
        if op == "pop":
            popped.append(stack.pop())
        else:
            stack.push(op)
    return (popped, len(stack))
```

**Why:** A `TypeVar` is a *link*, not a wildcard: it ties `push`'s argument to `pop`'s return value, so a checker knows `Stack[int]().pop()` is an `int` while `object` would have thrown that away. At runtime `Generic` does almost nothing — the subscript in `Stack[int]` is erased and no type is enforced — so this is a design tool, not a guard. Since 3.12 the same class is written `class Stack[T]:` with no `TypeVar` and no import; the older spelling is still what you will read in most code.

</details>

In [ ]:
from typing import Generic, TypeVar

T = TypeVar("T")


class Stack(Generic[T]):
    ...


def run_stack(ops):
    ...

check('Q-396', run_stack)

### Q-397 - L5 Hard - Concurrency

Write `bounded_gather(values, limit)` — a synchronous function that squares every value concurrently, but never lets more than `limit` coroutines be inside the work section at once.

Use an `asyncio.Semaphore(limit)` as an `async with`, have each coroutine `await asyncio.sleep(0.001)` while it holds the semaphore, and track the **peak** number of coroutines holding it simultaneously.

Return the tuple `(results, peak)` — `results` a **list** of squares in input order, `peak` an `int`. For an empty input return `([], 0)`.

Drive it all from `asyncio.run(...)`; `bounded_gather` itself is not `async`.

<details><summary>Hint</summary>

Keep a running `active` count and a `peak` in the enclosing coroutine and reach them with `nonlocal`. Increment right after acquiring and decrement before releasing — the peak then lands on `min(limit, len(values))`.

</details>

<details><summary>Solution</summary>

```python
import asyncio


def bounded_gather(values, limit):
    async def main():
        sem = asyncio.Semaphore(limit)
        active = 0
        peak = 0

        async def work(value):
            nonlocal active, peak
            async with sem:
                active += 1
                peak = max(peak, active)
                await asyncio.sleep(0.001)
                active -= 1
                return value * value

        results = await asyncio.gather(*(work(v) for v in values))
        return (results, peak)

    return asyncio.run(main())
```

**Why:** This is the shape of every real concurrent client: `gather` alone would fire ten thousand requests at once and get you rate-limited or out of sockets, so the semaphore turns 'all of them' into 'at most N of them' without changing the result or its order. Note that `active += 1` needs no lock here — a coroutine only yields at an `await`, so everything between two awaits is atomic, which is precisely the property threads do not give you (Q-391). `async with` releases the semaphore even when the body raises.

</details>

In [ ]:
import asyncio


def bounded_gather(values, limit):
    ...

check('Q-397', bounded_gather)

### Q-398 - L5 Hard - Testing

Write `run_suite(func, cases)` — a miniature test runner.

`cases` is a list of `(args_tuple, expected)` pairs. For each one, call `func(*args_tuple)` and classify the outcome as the string `"PASS"` (the result equals `expected`), `"FAIL"` (it does not) or `"ERROR"` (the call raised).

Return the tuple `(outcomes, summary)` where `outcomes` is the **list** of those strings in order and `summary` is a **plain `dict`** with the keys `"PASS"`, `"FAIL"` and `"ERROR"` — all three always present, counting `0` where nothing landed.

A raising case must never stop the run.

<details><summary>Hint</summary>

`try` / `except Exception` / `else` gives you the three branches cleanly: the comparison belongs in `else`, so a `False` comparison is never confused with a raised one. Seed the summary with all three keys at zero before you start counting.

</details>

<details><summary>Solution</summary>

```python
def run_suite(func, cases):
    summary = {"PASS": 0, "FAIL": 0, "ERROR": 0}
    outcomes = []
    for args, expected in cases:
        try:
            got = func(*args)
        except Exception:
            outcome = "ERROR"
        else:
            outcome = "PASS" if got == expected else "FAIL"
        outcomes.append(outcome)
        summary[outcome] += 1
    return (outcomes, summary)
```

**Why:** Separating FAIL from ERROR is the distinction every real runner makes and beginners collapse: a wrong answer means your code is wrong, an exception usually means your *test* is wrong (bad arguments, missing fixture) and they want different first moves. Catching `Exception` rather than bare `except:` is what lets Ctrl-C still stop the suite, and running every case before reporting is why pytest shows you thirty failures at once instead of making you rerun thirty times.

</details>

In [ ]:
def run_suite(func, cases):
    ...

check('Q-398', run_suite)

### Q-399 - L5 Hard - Debugging

Write `last_frame(text)` that reads a traceback the way you do — bottom up — and reports where it actually broke.

Frame lines look like `  File "app.py", line 6, in main` (always indented). The exception line is the last unindented line and looks like `KeyError: 'gamma'`.

Return a **plain `dict`**:

```python
{'file': 'app.py', 'line': 6, 'func': 'main',
 'type': 'KeyError', 'message': "'gamma'"}
```

from the **last** frame and the exception line. `line` is an `int`; the rest are strings. Return `None` if there is no frame or no exception line. Source-echo and `^^^^` marker lines are indented, so they must not be mistaken for either.

<details><summary>Hint</summary>

Two compiled patterns and one pass over `text.splitlines()`. Keep the most recent frame match rather than the first. The header line `Traceback (most recent call last):` is unindented too — make sure your exception pattern cannot match it.

</details>

<details><summary>Solution</summary>

```python
import re

FRAME_RE = re.compile(
    r'^\s+File "(?P<file>[^"]+)", line (?P<line>\d+), in (?P<func>.+)$'
)
EXC_RE = re.compile(r"^(?P<type>[A-Za-z_][\w.]*): (?P<message>.*)$")


def last_frame(text):
    frame = None
    exc = None
    for raw in text.splitlines():
        found = FRAME_RE.match(raw)
        if found is not None:
            frame = found
            continue
        if not raw.startswith(" "):
            hit = EXC_RE.match(raw)
            if hit is not None:
                exc = hit
    if frame is None or exc is None:
        return None
    return {
        "file": frame["file"],
        "line": int(frame["line"]),
        "func": frame["func"],
        "type": exc["type"],
        "message": exc["message"],
    }
```

**Why:** Tracebacks are printed oldest-call-first, so 'the last frame wins' is the parsing rule that matches how you read one — and it is why log aggregators group errors by the bottom frame rather than the top. The indentation test is doing real work: it is the only thing separating the exception line from the echoed source and `~~~^^^` markers that sit between frames, and the `Traceback (most recent call last):` header is excluded for free because `Traceback` is not followed by `: `. In production, reach for the `traceback` module (`traceback.extract_tb`) instead of parsing text — but when all you have is a log file, this is the job.

</details>

In [ ]:
import re


def last_frame(text):
    ...

check('Q-399', last_frame)

### Q-400 - L5 Hard - Functools

The last question of the course: rebuild `functools.lru_cache` from scratch.

Write `memoize(maxsize)`, a decorator factory. The wrapper it returns caches results keyed on the positional arguments tuple, evicts the **least recently used** entry once the cache exceeds `maxsize`, and counts hits and misses. A *use* means a hit as well as a miss. Give the wrapper a `cache_info()` method returning the tuple `(hits, misses, maxsize, currsize)`, and preserve the wrapped function's metadata with `functools.wraps`.

Then write `run_cached(maxsize, calls)`: define `square(n)` inside it, decorate it with `memoize(maxsize)`, call it once for each value in `calls` in order, and return the 4-tuple `(results, hits, misses, currsize)` — `results` a **list**.

`run_cached(1, [5, 5, 5])` returns `([25, 25, 25], 2, 1, 1)`. You may not call `lru_cache`.

<details><summary>Hint</summary>

`collections.OrderedDict` gives you insertion order plus two methods that make LRU a two-liner: one moves a key to the most-recent end, the other pops from the least-recent end. Record the hit *before* moving the key, and evict only after inserting — check the size, do not pre-empt it.

</details>

<details><summary>Solution</summary>

```python
from collections import OrderedDict
from functools import wraps


def memoize(maxsize):
    def decorate(func):
        cache = OrderedDict()
        stats = {"hits": 0, "misses": 0}

        @wraps(func)
        def wrapper(*args):
            if args in cache:
                stats["hits"] += 1
                cache.move_to_end(args)
                return cache[args]
            stats["misses"] += 1
            result = func(*args)
            cache[args] = result
            if len(cache) > maxsize:
                cache.popitem(last=False)
            return result

        def cache_info():
            return (stats["hits"], stats["misses"], maxsize, len(cache))

        wrapper.cache_info = cache_info
        return wrapper

    return decorate


def run_cached(maxsize, calls):
    @memoize(maxsize)
    def square(n):
        return n * n

    results = [square(n) for n in calls]
    hits, misses, _, currsize = square.cache_info()
    return (results, hits, misses, currsize)
```

**Why:** Three ideas meet here, and they are the three that make decorators worth learning. The closure is the storage: `cache` and `stats` live in `decorate`'s frame, one copy per decorated function, which is why you never needed a class. `OrderedDict.move_to_end` plus `popitem(last=False)` is the entire LRU policy — recency is just position, so eviction is O(1). And `@wraps` copies `__name__`, `__doc__` and `__wrapped__` across, without which every decorated function in your program reports itself as `wrapper` and `help()` goes blank. The real `lru_cache` adds thread safety, keyword arguments folded into the key, and a C implementation — but this is its shape, and you have now written it.

</details>

In [ ]:
from collections import OrderedDict
from functools import wraps


def memoize(maxsize):
    ...


def run_cached(maxsize, calls):
    ...

check('Q-400', run_cached)

---

## Mini-project P-10 - Log summariser

Build `parse_log(lines)` — the tool you actually reach for when a service misbehaves.

`lines` is a list of log lines, each shaped like:

```
2024-03-05 14:07:23 ERROR /srv/app/auth.py Login failed for user 7
```

that is: ISO date, space, `HH:MM:SS`, space, an uppercase level, space, a POSIX source path, space, then the message (which may contain spaces).

Parse each line with **one compiled regex** and reduce the path to its filename with `pathlib.PurePosixPath`. Return a **plain `dict`**:

```python
{'total': 3,
 'skipped': 0,
 'by_level': {'ERROR': 2, 'INFO': 1},
 'by_hour': {14: 2, 15: 1},
 'by_file': {'auth.py': 2, 'db.py': 1},
 'first_error': 'Login failed for user 7'}
```

Rules:

- `total` counts lines that parsed; `skipped` counts non-blank lines that did not.
- Blank or whitespace-only lines are ignored entirely — neither counted.
- `by_hour` keys are **`int`** hours `0..23`, taken from the timestamp.
- `by_level` and `by_file` contain only what was actually seen; all three sub-dicts are plain `dict`s, not `Counter`s.
- `first_error` is the message of the first `ERROR` line, or `None` if there is none.
- Nothing touches the filesystem, and nothing calls `now()`.

<details><summary>Hint</summary>

Named groups make the body readable: `date`, `time`, `level`, `path`, `message`. Strip each line first, skip it if it is empty, then `match` — anchoring at the start is what you want here. Count with three `Counter`s and convert them all to plain dicts in the return statement. The hour is `int(time[:2])`.

</details>

<details><summary>Solution</summary>

```python
import re
from collections import Counter
from pathlib import PurePosixPath

LINE_RE = re.compile(
    r"^(?P<date>\d{4}-\d{2}-\d{2}) "
    r"(?P<time>\d{2}:\d{2}:\d{2}) "
    r"(?P<level>[A-Z]+) "
    r"(?P<path>\S+) "
    r"(?P<message>.*)$"
)


def parse_log(lines):
    levels = Counter()
    hours = Counter()
    files = Counter()
    skipped = 0
    first_error = None

    for raw in lines:
        line = raw.strip()
        if not line:
            continue
        found = LINE_RE.match(line)
        if found is None:
            skipped += 1
            continue
        levels[found["level"]] += 1
        hours[int(found["time"][:2])] += 1
        files[PurePosixPath(found["path"]).name] += 1
        if found["level"] == "ERROR" and first_error is None:
            first_error = found["message"]

    return {
        "total": sum(levels.values()),
        "skipped": skipped,
        "by_level": dict(levels),
        "by_hour": dict(hours),
        "by_file": dict(files),
        "first_error": first_error,
    }
```

**Why:** This is the whole notebook in one function: a compiled regex with named groups does the parsing, `Counter` does the aggregation, `PurePosixPath` does the path surgery without ever touching a disk, and the function stays **pure** — lines in, dict out. That purity is the design decision worth keeping. Every real version of this tool grows an `argparse` front end and an `open()`, and the moment those live inside the parsing function the whole thing becomes untestable. Keep the I/O in a thin shell at the edge and the logic in a function you can call with a list of strings, and you get a test suite that runs in milliseconds with no fixtures at all. The `skipped` counter is the other habit: real logs always contain lines your pattern does not cover, and counting them turns a silent gap into a number you can watch.

</details>

In [ ]:
import re
from collections import Counter
from pathlib import PurePosixPath


def parse_log(lines):
    ...

check('P-10', parse_log)

---

## Your progress

In [ ]:
progress()